In [1]:
from dotenv import load_dotenv

from langchain_groq import ChatGroq

from langgraph.prebuilt import create_react_agent

from pathlib import Path
import pandas as pd

In [2]:
load_dotenv()

print("Environment variables loaded.")

Environment variables loaded.


In [3]:
data_path = Path("../data/structured")

employees = pd.read_csv(
    data_path / "employees.csv"
)

attendance = pd.read_csv(
    data_path / "attendance.csv"
)

leave = pd.read_csv(
    data_path / "leave.csv"
)

holidays = pd.read_csv(
    data_path / "holidays.csv"
)

attendance["date"] = pd.to_datetime(
    attendance["date"]
)

leave["start_date"] = pd.to_datetime(
    leave["start_date"]
)

leave["end_date"] = pd.to_datetime(
    leave["end_date"]
)

holidays["date"] = pd.to_datetime(
    holidays["date"]
)

print("Structured data loaded.")

Structured data loaded.


In [4]:
from langchain_core.tools import tool

In [5]:
@tool
def get_employee(employee_id: str) -> str:
    """
    Get information about an employee.

    Use this tool when you need information such as
    employee name, department, role, employment type,
    weekly working hours, vacation entitlement,
    or office location.
    """

    result = employees[
        employees["employee_id"] == employee_id
    ]

    if result.empty:
        return f"No employee found with ID {employee_id}."

    employee = result.iloc[0]

    return (
        f"Employee ID: {employee['employee_id']}\n"
        f"Name: {employee['name']}\n"
        f"Department: {employee['department']}\n"
        f"Role: {employee['role']}\n"
        f"Employment type: {employee['employment_type']}\n"
        f"Weekly hours: {employee['weekly_hours']}\n"
        f"Vacation entitlement: {employee['vacation_days']} days\n"
        f"Office location: {employee['office_location']}"
    )

In [6]:
@tool
def get_attendance_summary(
    employee_id: str,
    start_date: str,
    end_date: str
) -> str:
    """
    Get attendance statistics for an employee
    during a specified date range.

    Returns office days, home-office days,
    business-trip days, sick days, leave days,
    and missing attendance records.
    """

    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)

    records = attendance[
        (attendance["employee_id"] == employee_id) &
        (attendance["date"] >= start) &
        (attendance["date"] <= end)
    ]

    if records.empty:
        return (
            f"No attendance records found for "
            f"{employee_id} between "
            f"{start_date} and {end_date}."
        )

    office_days = (
        records["location"] == "office"
    ).sum()

    home_days = (
        records["location"] == "home"
    ).sum()

    business_trip_days = (
        records["location"] == "business_trip"
    ).sum()

    sick_days = (
        records["status"] == "sick"
    ).sum()

    leave_days = (
        records["status"] == "leave"
    ).sum()

    missing_records = records[
        records["status"].isin(
            ["missing", "missing_checkout"]
        )
    ].shape[0]

    return (
        f"Employee: {employee_id}\n"
        f"Period: {start_date} to {end_date}\n"
        f"Office days: {office_days}\n"
        f"Home-office days: {home_days}\n"
        f"Business-trip days: {business_trip_days}\n"
        f"Sick days: {sick_days}\n"
        f"Leave days: {leave_days}\n"
        f"Missing attendance records: {missing_records}"
    )

In [7]:
@tool
def calculate_working_hours(
    employee_id: str,
    start_date: str,
    end_date: str
) -> str:
    """
    Calculate total recorded working hours
    for an employee during a date range.
    """

    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)

    records = attendance[
        (attendance["employee_id"] == employee_id) &
        (attendance["date"] >= start) &
        (attendance["date"] <= end)
    ].copy()

    records = records[
        records["check_in"].notna() &
        records["check_out"].notna() &
        (records["check_in"] != "") &
        (records["check_out"] != "")
    ]

    if records.empty:
        return (
            f"No complete attendance records found "
            f"for {employee_id}."
        )

    records["check_in_time"] = pd.to_datetime(
        records["check_in"],
        format="%H:%M"
    )

    records["check_out_time"] = pd.to_datetime(
        records["check_out"],
        format="%H:%M"
    )

    records["hours"] = (
        records["check_out_time"]
        - records["check_in_time"]
    ).dt.total_seconds() / 3600

    total_hours = records["hours"].sum()

    return (
        f"Employee: {employee_id}\n"
        f"Period: {start_date} to {end_date}\n"
        f"Total recorded working hours: "
        f"{total_hours:.2f}"
    )

In [8]:
@tool
def find_missing_attendance(
    employee_id: str,
    start_date: str,
    end_date: str
) -> str:
    """
    Find dates where an employee has missing
    attendance or a missing check-out.
    """

    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)

    records = attendance[
        (attendance["employee_id"] == employee_id) &
        (attendance["date"] >= start) &
        (attendance["date"] <= end)
    ]

    missing = records[
        records["status"].isin(
            ["missing", "missing_checkout"]
        )
    ]

    if missing.empty:
        return (
            f"No missing attendance records found "
            f"for {employee_id} between "
            f"{start_date} and {end_date}."
        )

    results = []

    for _, row in missing.iterrows():

        results.append(
            f"{row['date'].date()} - "
            f"{row['status']}"
        )

    return (
        f"Missing attendance for {employee_id}:\n"
        + "\n".join(results)
    )

In [9]:
@tool
def get_leave_balance(
    employee_id: str
) -> str:
    """
    Get an employee's vacation entitlement,
    approved vacation used, and remaining vacation.
    """

    employee_result = employees[
        employees["employee_id"] == employee_id
    ]

    if employee_result.empty:
        return f"No employee found with ID {employee_id}."

    employee = employee_result.iloc[0]

    entitlement = int(
        employee["vacation_days"]
    )

    approved_vacation = leave[
        (leave["employee_id"] == employee_id) &
        (leave["type"] == "vacation") &
        (leave["status"] == "approved")
    ]

    used = int(
        approved_vacation["days"].sum()
    )

    remaining = entitlement - used

    return (
        f"Employee: {employee_id}\n"
        f"Vacation entitlement: {entitlement} days\n"
        f"Approved vacation used: {used} days\n"
        f"Remaining vacation: {remaining} days"
    )

In [10]:
tools = [
    get_employee,
    get_attendance_summary,
    calculate_working_hours,
    find_missing_attendance,
    get_leave_balance
]

for tool in tools:
    print(tool.name)

get_employee
get_attendance_summary
calculate_working_hours
find_missing_attendance
get_leave_balance


In [11]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    reasoning_format="parsed"
)

print("LLM ready.")

LLM ready.


In [12]:
agent = create_react_agent(
    model=llm,
    tools=tools
)

print("Agent created successfully.")

Agent created successfully.


/var/folders/kb/3rhgzw5d6md74w5kr02cc4cm0000gn/T/ipykernel_1845/4204495772.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [13]:
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "How many days did E0001 work from home in August 2026?"
            }
        ]
    }
)

print(
    response["messages"][-1].content
)

E0001 worked from home **5 days** in August 2026.


In [14]:
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "How much vacation does E0001 have left?"
            }
        ]
    }
)

print(
    response["messages"][-1].content
)

E0001 has **25 vacation days** remaining.


In [15]:
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "How many hours did E0001 work in August 2026?"
            }
        ]
    }
)

print(
    response["messages"][-1].content
)

E0001 logged **148.65 hours** of work during August 2026.


In [16]:
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Did E0001 have any missing attendance records in August 2026?"
            }
        ]
    }
)

print(
    response["messages"][-1].content
)

Yes. E0001 had three days with attendance issues in August 2026:

| Date | Issue |
|------|-------|
| 2026‑08‑11 | Missing attendance record |
| 2026‑08‑14 | Missing checkout (clock‑out) |
| 2026‑08‑31 | Missing attendance record |

So, there were missing attendance entries on those three days.
